## Basic use case
cmd: `python main.py`

In [ ]:
from dataclasses import dataclass
from hydra.core.config_store import ConfigStore
import hydra
from omegaconf import OmegaConf, DictConfig

@dataclass
class ExperimentConfig:
    model: str = "resnet18"
    nrof_epochs: int=30
    learning_rate: float=5e-3

cs = ConfigStore.instance()
cs.store(name='config', node=ExperimentConfig)

@hydra.main(config_path=None, config_name='config', version_base=None)
def main(config: DictConfig) -> None:
    print(OmegaConf.to_yaml(config))

if __name__ == "__main__":
    main()

## Hierarchical Static Config

cmd: `python main.py`

In [ ]:
from dataclasses import dataclass

@dataclass
class ExperimentConfig:
    model: str = "resnet18"
    nrof_epochs: int = 30
    learning_rate: float = 5e-3

    def __post_init__(self):
        print(f"ExperimentConfig instance created at: {id(self)}")

@dataclass
class LossConfig:
    name: str = "arcface"
    margin: float = 0.5

    def __post_init__(self):
        print(f"LossConfig instance created at: {id(self)}")

@dataclass
class MyConfig:
    training: ExperimentConfig = ExperimentConfig()
    loss: LossConfig = LossConfig()

print("Creating config_1:")
config_1 = MyConfig()
print("Creating config_2:")
config_2 = MyConfig()

print("IDs of shared instances:")
print(f"config_1.training ID: {id(config_1.training)}")
print(f"config_2.training ID: {id(config_2.training)}")


# THIS IS THE OUTPUT IF IT DON'T RAISE ERROR
# 
# ExperimentConfig instance created at: 140089523418256
# LossConfig instance created at: 140089523418960
# Creating config_1:
# Creating config_2:
# IDs of shared instances:
# config_1.training ID: 140089523418256
# config_2.training ID: 140089523418256


In [ ]:
from dataclasses import dataclass, field
from omegaconf import DictConfig, OmegaConf
import hydra
from hydra.core.config_store import ConfigStore


@dataclass
class ExperimentConfig:
    model: str = "resnet18"
    nrof_epochs: int = 30
    learning_rate: float = 5e-3


@dataclass
class LossConfig:
    name: str = "arcface"
    margin: float = 0.5


@dataclass
class MyConfig:
    training: ExperimentConfig = field(
        default_factory=ExperimentConfig
    )  # ensure new instance of ExperimentConfig is created for new MyConfig instance
    loss: LossConfig = field(default_factory=LossConfig)  # same reason


cs = ConfigStore.instance()
cs.store(name="config", node=MyConfig)


@hydra.main(config_path=None, config_name="config", version_base=None)
def main(config: DictConfig) -> None:
    print(OmegaConf.to_yaml(config))


if __name__ == "__main__":
    main()


## Config group in structured config

cmd: `python main.py +experiment=resnet18`

output:
```
experiment:
  model: resnet18
  nrof_epochs: 18
  lr: 0.0005
```

In [ ]:
from dataclasses import dataclass
from typing import Any
from omegaconf import DictConfig, OmegaConf
import hydra
from hydra.core.config_store import ConfigStore

@dataclass
class ResNet18Exp:
    model: str = "resnet18"
    nrof_epochs: int = 18
    lr: float = 5e-4

@dataclass
class ResNet50Exp:
    model: str = "resnet50"
    nrof_epochs: int = 50
    lr: float = 5e-4

@dataclass
class MyConfig:
    experiment: Any

cs = ConfigStore.instance()
cs.store(name="config", node=MyConfig)
cs.store(group="experiment", name='resnet18', node=ResNet18Exp)
cs.store(group="experiment", name='resnet50', node=ResNet50Exp)


@hydra.main(config_path=None, config_name="config", version_base=None)
def main(config:DictConfig) -> None:
    print(OmegaConf.to_yaml(config))

if __name__ == "__main__":
    main()

## Inheritance in Structured Config
cmd: `python main.py +experiment=resnet18`

In [ ]:
from dataclasses import dataclass
from typing import Any
from omegaconf import DictConfig, OmegaConf, MISSING
import hydra
from hydra.core.config_store import ConfigStore

@dataclass
class Exp:
    model: str = MISSING
    nrof_epochs: int = 20
    lr: float = 5e-4

@dataclass
class ResNet18Exp(Exp):
    model: str = "resnet18"

@dataclass
class ResNet50Exp(Exp):
    model: str = "resnet50"

@dataclass
class MyConfig:
    experiment: Any

cs = ConfigStore.instance()
cs.store(name="config", node=MyConfig)
cs.store(group="experiment", name='resnet18', node=ResNet18Exp)
cs.store(group="experiment", name='resnet50', node=ResNet50Exp)


@hydra.main(config_path=None, config_name="config",version_base=None)
def main(config:DictConfig) -> None:
    print(OmegaConf.to_yaml(config))

if __name__ == "__main__":
    main()

## Structured Config Schema

```text
configs
├── config.yaml
└── experiment
    ├── resnet18.yaml
    └── resnet50.yaml
```

```yaml
# ./configs/config.yaml
defaults:
  - config_schema
  - experiment: resnet50
  - _self_
```

```yaml
# ./configs/experiment/resnet18.yaml

defaults:
  - resnet18_schema

model: resnet18
batch_size: 32
nrof_epochs: 30
```

```yaml
# ./configs/experiment/resnet50.yaml

defaults:
  - resnet50_schema

model: resnet50
batch_size: 64
nrof_epochs: 30
```

In [ ]:
# main.py
from dataclasses import dataclass
from typing import Any
from omegaconf import DictConfig, OmegaConf, MISSING
import hydra
from hydra.core.config_store import ConfigStore


@dataclass
class ExpSchema:
    model: str = MISSING
    nrof_epochs: int = 20
    lr: float = 5e-4
    batch_size: int = 512

@dataclass
class ResNet18ExpSchema(ExpSchema):
    model: str = "resnet18"

@dataclass
class ResNet50ExpSchema(ExpSchema):
    model: str = "resnet50"

@dataclass
class ConfigSchema:
    experiment: ExpSchema

cs = ConfigStore.instance()
cs.store(name="config_schema", node=ConfigSchema)
cs.store(group="experiment", name="resnet18_schema", node=ResNet18ExpSchema)
cs.store(group="experiment", name="resnet50_schema", node=ResNet50ExpSchema)

@hydra.main(config_path="configs", config_name="config", version_base=None)
def main(config: DictConfig) -> None:
    print(OmegaConf.to_yaml(config))


if __name__ == "__main__":
    main()


## Validating config parameters

In [ ]:
from typing import Any
from omegaconf import DictConfig, OmegaConf, MISSING
import hydra
from hydra.core.config_store import ConfigStore
from pydantic.dataclasses import dataclass
from pydantic import validator


@dataclass
class ExpSchema:
    model: str = MISSING
    nrof_epochs: int = 20
    lr: float = 5e-4
    batch_size: int = 512

    @validator("batch_size")
    def batch_size_multiple_of_32(cls, batch_size: int) -> int:
        if batch_size % 32 != 0:
            raise ValueError("batch_size should be multiple of 32")
        return batch_size


@dataclass
class ResNet18ExpSchema(ExpSchema):
    model: str = "resnet18"


@dataclass
class ResNet50ExpSchema(ExpSchema):
    model: str = "resnet50"


@dataclass
class ConfigSchema:
    experiment: ExpSchema


cs = ConfigStore.instance()
cs.store(name="config_schema", node=ConfigSchema)
cs.store(group="experiment", name="resnet18_schema", node=ResNet18ExpSchema)
cs.store(group="experiment", name="resnet50_schema", node=ResNet50ExpSchema)


@hydra.main(config_path="configs", config_name="config", version_base=None)
def main(config: DictConfig) -> None:
    OmegaConf.to_object(config)
    print(OmegaConf.to_yaml(config))


if __name__ == "__main__":
    main()
